# 🧠 SemanticDoc ML & RAG Interactive Refresher

Run each cell below using **`Shift + Enter`** to see how data transforms at every single stage of our pipeline:
1. **PyTorch (`torch`)**: Tensors & Hardware Acceleration
2. **Hugging Face (`sentence-transformers`)**: Turning English into 384-Dimensional Vectors
3. **LangChain (`PyPDFLoader` + `RecursiveCharacterTextSplitter`)**: Slicing a PDF into Overlapping Chunks
4. **FAISS (`faiss-cpu`)**: Indexing Vectors & Finding Closest Meaning (`R` = Retrieval)
5. **LangChain Prompts + Ollama (`llama3.2`)**: Grounded Answer Generation (`A` + `G` = Augmented Generation)

---
## Stage 1: PyTorch & Tensors
**What is a Tensor?** A multi-dimensional array of numbers that lives on CPU or GPU (`mps` / `cuda`). Neural networks only understand Tensors!

In [6]:
import torch

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"PyTorch Version: {torch.__version__} | Active Compute Device: '{device}'")

# A 1D Tensor (Vector) of 5 numbers:
sample_vector = torch.tensor([0.12, -0.45, 0.88, 0.03, -0.19], device=device)
print("Sample Tensor:", sample_vector)
print("Shape:", sample_vector.shape)

PyTorch Version: 2.15.0+cu132 | Active Compute Device: 'cpu'
Sample Tensor: tensor([ 0.1200, -0.4500,  0.8800,  0.0300, -0.1900])
Shape: torch.Size([5])


---
## Stage 2: Hugging Face Embedding Model (`all-MiniLM-L6-v2`)
**Role:** Acts as the **Translator** from English -> 384-number PyTorch Tensor.
- Sentences with similar *meaning* point in the same direction (high Cosine Similarity), even if they share zero words!

In [7]:
from sentence_transformers import SentenceTransformer, util

hf_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

sentences = [
    "Employees receive 20 days of paid vacation each year.",   # Meaning: Time off
    "Database passwords must be rotated every 90 days.",       # Meaning: IT Security
]
query = "How much time off do I get?"

sentence_tensors = hf_model.encode(sentences, convert_to_tensor=True)
query_tensor = hf_model.encode(query, convert_to_tensor=True)

print(f"Query Tensor Shape: {query_tensor.shape} (384 dimensions!)")
print(f"First 5 numbers of Query Tensor: {query_tensor[:5].tolist()}\n")

scores = util.cos_sim(query_tensor, sentence_tensors)[0]
for idx, text in enumerate(sentences):
    print(f"Cosine Similarity: {scores[idx]:.4f} | Text: {text}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query Tensor Shape: torch.Size([384]) (384 dimensions!)
First 5 numbers of Query Tensor: [0.05878140404820442, -0.017009111121296883, 0.007759305648505688, 0.020618753507733345, 0.0284462571144104]

Cosine Similarity: 0.2716 | Text: Employees receive 20 days of paid vacation each year.
Cosine Similarity: 0.1117 | Text: Database passwords must be rotated every 90 days.


---
## Stage 3: PDF Loading & Chunking (`LangChain`)
**Why Chunk?**
1. `all-MiniLM-L6-v2` has a hard limit of 256 tokens (~200 words).
2. Smaller chunks keep each 384D vector focused on a single topic, and preserve the **Page Number** in `metadata`!

In [8]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = "/app/sample_data/handbook.pdf" if os.path.exists("/app/sample_data/handbook.pdf") else "../sample_data/handbook.pdf"
loader = PyPDFLoader(pdf_path)
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    add_start_index=True,
)
chunks = splitter.split_documents(pages)

print(f"Split {len(pages)} PDF pages into {len(chunks)} chunks!\n")
for i, chunk in enumerate(chunks[:3]):  # Inspect first 3 chunks
    print(f"--- Chunk #{i} (Page {chunk.metadata['page'] + 1}) ---")
    print(chunk.page_content)
    print()

Split 3 PDF pages into 7 chunks!

--- Chunk #0 (Page 1) ---
GoLogic Engineering Handbook - Page 1: Work & Leave Policy
1. Remote Work: Engineers may work remotely up to 3 days per week.
Core collaboration hours are 10:00 AM to 3:00 PM AEST regardless of location.
2. Annual Leave & Vacation: Full-time employees accrue 20 days of paid annual leave

--- Chunk #1 (Page 1) ---
per year. Leave requests exceeding 5 consecutive days require Engineering Manager
approval at least two weeks in advance.
3. Home Office Stipend: Every new hire receives a one-time $1,200 AUD allowance
to purchase an ergonomic chair, monitor, or mechanical keyboard.

--- Chunk #2 (Page 2) ---
GoLogic Engineering Handbook - Page 2: Security & Infrastructure
4. Secret Management: Production database credentials are managed via HashiCorp Vault.
All database passwords and API keys automatically rotate every 90 days.



---
## Stage 4: ”R” in RAG — Vector Database & Retrieval (`FAISS`)
**Role:**
- **PyTorch** converts the chunks into 384D vectors.
- **FAISS** stores those vectors on disk and searches for the **Top $K$ closest chunks** using L2 Distance (**Lower = Closer match!**).

In [9]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

# Build FAISS index in memory from our PDF chunks
vector_store = FAISS.from_documents(chunks, embeddings)

# Test a retrieval query!
user_question = "How much is the home office stipend and what can I buy?"
retrieved_docs = vector_store.similarity_search_with_score(user_question, k=3)

print(f"Question: '{user_question}'\n")
for rank, (doc, l2_dist) in enumerate(retrieved_docs, start=1):
    print(f"Rank #{rank} | L2 Distance: {l2_dist:.4f} | Page {doc.metadata['page'] + 1}")
    print(f"Content: {doc.page_content}\n")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Question: 'How much is the home office stipend and what can I buy?'

Rank #1 | L2 Distance: 1.1420 | Page 1
Content: per year. Leave requests exceeding 5 consecutive days require Engineering Manager
approval at least two weeks in advance.
3. Home Office Stipend: Every new hire receives a one-time $1,200 AUD allowance
to purchase an ergonomic chair, monitor, or mechanical keyboard.

Rank #2 | L2 Distance: 1.7025 | Page 1
Content: GoLogic Engineering Handbook - Page 1: Work & Leave Policy
1. Remote Work: Engineers may work remotely up to 3 days per week.
Core collaboration hours are 10:00 AM to 3:00 PM AEST regardless of location.
2. Annual Leave & Vacation: Full-time employees accrue 20 days of paid annual leave

Rank #3 | L2 Distance: 1.8090 | Page 3
Content: GoLogic Engineering Handbook - Page 3: Incident Response
6. Severity 1 (SEV-1) Outages: A SEV-1 incident is defined as complete unavailability
of the core payment API affecting more than 5% of active users.
The on-call engineer mu

---
## Stage 5: ”A” & ”G” in RAG — Augmented Generation (`Ollama LLM`)
**Role:**
- **A (Augmented):** `ChatPromptTemplate` stuffs the Top 3 chunks from FAISS + the user question into one strict prompt.
- **G (Generation):** `ChatOllama` (`llama3.2:1b`) reads those chunks, reasons over them, ignores irrelevant noise, and writes a cited human answer!

In [10]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model=os.getenv("OLLAMA_MODEL", "llama3.2:1b"),
    base_url=os.getenv("OLLAMA_BASE_URL", "http://host.docker.internal:11434"),
    temperature=0.0,
)

prompt_template = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful assistant that answers questions based strictly on the provided handbook excerpts. "
        "Always mention the Page Number in your answer."
    ),
    (
        "human",
        "Read the following excerpts from the GoLogic Engineering Handbook:\n\n"
        "{context}\n\n"
        "Question: {question}\n"
        "Instruction: Answer the question using the excerpts above. "
        "Only if the excerpts are completely unrelated to the question, reply: "
        "'I do not have enough information in the document to answer that.'"
    ),
])

# Format the retrieved chunks from Stage 4
formatted_context = "\n\n".join(
    f"[Excerpt from Page {doc.metadata['page'] + 1}]: {doc.page_content}"
    for doc, _ in retrieved_docs
)

messages = prompt_template.invoke({
    "context": formatted_context,
    "question": user_question,
})

response = llm.invoke(messages)
print("🤖 FINAL RAG ANSWER:\n" + "-" * 40)
print(response.content)

🤖 FINAL RAG ANSWER:
----------------------------------------
Based on the excerpts provided, the home office stipend is $1,200 AUD. This stipend is mentioned on Page 1 of the GoLogic Engineering Handbook.

The excerpts do not mention what you can buy with the home office stipend. They only provide information about the stipend itself.
